# `lut-hashes-accessor` — pre-merge test notebook

Branch: `lut-hashes-accessor` &nbsp;|&nbsp; commit `09ef368` &nbsp;|&nbsp; gate **M1**

**What this branch adds:** `evoca_get_lut_hashes()` / `sim.get_lut_hashes()` — a per-cell
FNV-1a hash of the **LUT bytes only**, distinct from the *combined* LUT‖egene genome
hash that the activity tracker / `lineage_parent_hash` expose. #8 (direct lineage
co-evolution metric) needs to decompose a parent→child change into ΔLUT independently
of Δegene; that requires a LUT-only hash.

**Pre-merge checklist (run all cells top-to-bottom; every cell prints `PASS`):**
1. Accessor loads, returns the right shape/dtype, a uniform LUT hashes uniformly.
2. **Egene-invariance** — changing only the egenome must not move any lut-hash. *(This is
   the property the combined hash fails; it is the entire reason the accessor exists.)*
3. **LUT-sensitivity + correctness** — changing the LUT changes every hash, matching an
   independent Python re-implementation of the C `lut_hash_fn`.
4. **Live-step invariant** — under GoL with no mutation the accessor stays consistent
   across `step()`s (it tracks the live lattice, no drift/corruption).

Mirrors `tests/test_lut_hashes.py` interactively. If anything prints `FAIL`, do not merge.

In [ ]:
import os, sys
import numpy as np

ROOT = os.getcwd()
sys.path.insert(0, os.path.join(ROOT, 'python'))
from evoca_py import EvoCA, make_gol_lut, LUT_BYTES

sim = EvoCA()
sim.init(16)
print('repo root :', ROOT)
print('lib       :', sim._lib._name if hasattr(sim._lib, '_name') else '(loaded)')
print('has accessor:', hasattr(sim, 'get_lut_hashes'))
assert hasattr(sim, 'get_lut_hashes'), 'sim.get_lut_hashes missing — wrong branch / stale dylib'
print('PASS — setup')

In [ ]:
# 1. shape / dtype / uniform-LUT-hashes-uniformly
gol = make_gol_lut()
sim.set_lut_all(gol)
sim.set_egenome_all(0b000011)

h = sim.get_lut_hashes()
print('shape', h.shape, 'dtype', h.dtype, 'distinct', len(np.unique(h)))
assert h.shape == (16, 16) and h.dtype == np.uint32
assert len(np.unique(h)) == 1, 'uniform GoL LUT should give one distinct hash'
print('GoL lut-hash = 0x%08x' % h.flat[0])
print('PASS — basic')

In [ ]:
# 2. EGENE-INVARIANCE — the load-bearing property
sim.set_lut_all(gol)
sim.set_egenome_all(0b000011)
h0 = sim.get_lut_hashes()

sim.set_egenome_all(0b111111)   # change ONLY the egenome
h1 = sim.get_lut_hashes()

same = np.array_equal(h0, h1)
print('lut-hash identical after egene change:', same)
assert same, 'FAIL — lut-hash leaked egene state (NOT LUT-only); do not merge'
print('PASS — egene-invariance (this is what the combined hash fails)')

In [ ]:
# 3. LUT-SENSITIVITY + CORRECTNESS vs an independent Python FNV mirror
def fnv1a_lut(lut_bytes):
    """Python mirror of C lut_hash_fn: FNV-1a over LUT_BYTES/4 LE uint32 words."""
    w = np.frombuffer(bytes(bytearray(lut_bytes[:LUT_BYTES])), dtype='<u4')
    hh = 0x811c9dc5
    for x in w:
        hh = ((hh ^ int(x)) * 0x01000193) & 0xFFFFFFFF
    return hh

sim.set_lut_all(gol)
h_gol = sim.get_lut_hashes()
assert np.all(h_gol == fnv1a_lut(gol)), 'FAIL — GoL hash != Python FNV mirror'

rng = np.random.default_rng(0)
rnd = rng.integers(0, 256, size=LUT_BYTES, dtype=np.uint8)
sim.set_lut_all(rnd)
h_rnd = sim.get_lut_hashes()

assert np.all(h_rnd != h_gol), 'FAIL — hash did not respond to LUT change'
assert np.all(h_rnd == fnv1a_lut(rnd)), 'FAIL — random-LUT hash != Python FNV mirror'
print('GoL  -> 0x%08x  (mirror 0x%08x)' % (h_gol.flat[0], fnv1a_lut(gol)))
print('rand -> 0x%08x  (mirror 0x%08x)' % (h_rnd.flat[0], fnv1a_lut(rnd)))
print('PASS — LUT-sensitivity + correctness')

In [ ]:
# 4. LIVE-STEP INVARIANT — accessor tracks the live lattice, no drift under GoL/no-mutation
s2 = EvoCA()
s2.init(32)
s2.set_lut_all(make_gol_lut())
s2.set_egenome_all(0)
s2.set_alive_all()
s2.update_mu_lut(0.0)
base = s2.get_lut_hashes().copy()
for _ in range(20):
    s2.step()
after = s2.get_lut_hashes()
stable = np.array_equal(base, after)
print('lut-hash stable over 20 GoL steps (mu_lut=0):', stable, '| distinct =', len(np.unique(after)))
assert stable, 'FAIL — lut-hash drifted with no LUT mutation'
print('PASS — live-step invariant')

## Expected result

All five cells print `PASS`. Key guarantees demonstrated:

| Cell | Guarantee | Why it matters for #8 |
|---|---|---|
| 2 | uniform LUT → one hash; right shape/dtype | sane accessor |
| 3 | **egene change → hash unchanged** | the LUT-only property the combined hash lacks |
| 4 | LUT change → hash changes, == Python FNV mirror | correctness (no approximation) |
| 5 | stable across `step()` with `mu_lut=0` | tracks live lattice, no corruption |

If all `PASS`, the accessor is purely additive and behaviour-neutral — safe to merge (M1).
The full pytest guard is `tests/test_lut_hashes.py`; the regression evidence is the
unchanged 40-test suite + clean `-Wall` build noted in the commit.